In [332]:
from typing import Optional
from DATA.stock_invest_function import get_db_host
import pymysql
import pandas as pd
import numpy as np
def fetch_fs_data_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = "korea_fs_data_from_DART") -> pd.DataFrame:
    """
    특정 ticker의 재무제표 데이터를 DB에서 조회.
    ticker가 존재하지 않을 경우 메시지 출력 후 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 먼저 ticker 존재 여부 확인
        check_sql = f"SELECT COUNT(*) AS cnt FROM {table_name} WHERE ticker = %s"
        with conn.cursor() as cur:
            cur.execute(check_sql, (ticker,))
            result = cur.fetchone()
            cnt = result[0]

        if cnt == 0:
            print(f"[INFO] ticker '{ticker}' 는(은) 데이터베이스에 존재하지 않습니다.")
            return pd.DataFrame()   # 빈 DF 반환

        # ticker 존재 → 실제 데이터 조회
        query = f"""
            SELECT
                corp_code,
                bsns_year,
                reprt_code,
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY
                bsns_year,
                reprt_code,
                sj_div,
                account_nm
        """

        df = pd.read_sql(query, conn, params=[ticker])
        return df

    finally:
        conn.close()

def adjust_quarterly_from_index(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    original_index_name = df.index.name

    df["year"] = df.index.year
    df["quarter"] = df.index.month.map({3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"})

    adjusted_chunks = []

    for year, grp in df.groupby("year"):
        grp = grp.sort_index()

        q1 = grp[grp["quarter"] == "Q1"]
        q2 = grp[grp["quarter"] == "Q2"]
        q3 = grp[grp["quarter"] == "Q3"]
        q4 = grp[grp["quarter"] == "Q4"]

        if len(q4) > 0:
            q4 = q4.copy()

            for col in value_cols:
                if col not in grp.columns:
                    continue

                fy = q4[col].iloc[0]  # 12월 값(FY라고 가정)
                if pd.isna(fy):
                    continue

                prev_sum = (
                    q1[col].fillna(0).sum()
                    + q2[col].fillna(0).sum()
                    + q3[col].fillna(0).sum()
                )

                q4[col] = fy - prev_sum   # ★ 여기서 딱 Q4만 수정

            adjusted_chunks.extend([q1, q2, q3, q4])
        else:
            # 4Q(12월)가 없으면 그 연도는 그대로
            adjusted_chunks.append(grp)

    result = pd.concat(adjusted_chunks).sort_index()
    result = result.drop(columns=["year", "quarter"])
    result.index.name = original_index_name
    return result


from typing import Optional, List

def cumulative_to_quarterly(df: pd.DataFrame,
                            value_cols: List[str],
                            exclude_date: Optional[str] = "2025-12-31") -> pd.DataFrame:
    """
    연도별 누적값(1Q,2Q,3Q,4Q)을 순수 분기값으로 변환.
    df: index가 날짜(분기말)인 DataFrame
    value_cols: 변환할 숫자 컬럼 리스트
    exclude_date: 제외할 날짜 (예: '2025-12-31')
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out = out.sort_index()

    # 특정 날짜 제거
    if exclude_date is not None:
        out = out.loc[out.index != pd.to_datetime(exclude_date)].copy()

    years = out.index.year

    for col in value_cols:
        def _to_quarterly(s: pd.Series) -> pd.Series:
            s = s.sort_index()
            q = s.diff()
            if len(s) > 0:
                q.iloc[0] = s.iloc[0]
            return q

        out[col] = (
            out[col]
            .groupby(years)
            .apply(_to_quarterly)
            .reset_index(level=0, drop=True)
        )

    return out


# 0으로 나누는 경우 inf가 생기지 않도록 float 변환 + 나누기 후 정리
def safe_divide(num, den):
    result = num / den
    # 0으로 나눠서 생긴 inf/-inf 를 NaN으로 처리
    result = result.replace([np.inf, -np.inf], np.nan)
    return result


In [333]:
import pymysql
import pandas as pd
import numpy as np
from typing import Optional, List

# ============================================================
# 0) DB 정보
# ============================================================
db_info = {
    'host': get_db_host(),   # 교수님이 이미 정의한 함수라고 가정
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "korea_fs_data_from_DART"


# ============================================================
# 1) 유틸 함수들
# ============================================================

def safe_divide(num: pd.Series, den: pd.Series) -> pd.Series:
    """0 나누기 등에서 생기는 inf/-inf 를 NaN으로 처리하는 안전한 나누기"""
    result = num / den
    result = result.replace([np.inf, -np.inf], np.nan)
    return result


def safe_series(df: pd.DataFrame, col: str, dtype: str = "float64") -> pd.Series:
    """
    df에 col 이 있으면 해당 Series를, 없으면
    같은 index를 가진 NaN Series를 반환.
    항상 pd.Series 를 반환하므로 .fillna()를 안전하게 쓸 수 있음.
    """
    if col in df.columns:
        return df[col].copy()
    else:
        return pd.Series(index=df.index, dtype=dtype)


def cumulative_to_quarterly(df: pd.DataFrame,
                            value_cols: List[str]) -> pd.DataFrame:
    """
    연도별 누적값(1Q,2Q,3Q,4Q)을 순수 분기값으로 변환.
    - df: index가 날짜(분기말)인 DataFrame (datetime index)
    - value_cols: 변환할 수치 컬럼 리스트
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out = out.sort_index()

    years = out.index.year

    for col in value_cols:
        if col not in out.columns:
            continue

        def _to_quarterly(s: pd.Series) -> pd.Series:
            s = s.sort_index()
            q = s.diff()
            if len(s) > 0:
                q.iloc[0] = s.iloc[0]  # 해당 연도 첫 분기는 그대로
            return q

        out[col] = (
            out[col]
            .groupby(years)
            .apply(_to_quarterly)
            .reset_index(level=0, drop=True)
        )

    return out


def adjust_quarterly_from_index(df: pd.DataFrame,
                                value_cols: List[str]) -> pd.DataFrame:
    """
    IS 등의 누적 계정을 분기값으로 변환.
    구조는 cumulative_to_quarterly 와 동일하게 구현.
    (이름만 교수님 기존 코드와 맞추기 위해 둠)
    """
    return cumulative_to_quarterly(df, value_cols)


# ============================================================
# 2) DB에서 전체 재무데이터 한번에 읽기
# ============================================================

def fetch_all_fs_data(db_info: dict,
                      table_name: str = TABLE_NAME) -> pd.DataFrame:
    """
    korea_fs_data_from_DART 전체를 한 번에 읽어서 반환.
    필요한 최소 컬럼만 가져와서 속도/메모리 절약.
    """
    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info['port'],
        user=db_info['user'],
        password=db_info['password'],
        database=db_info['database'],
        charset='utf8mb4'
    )
    try:
        query = f"""
            SELECT
                corp_code,
                ticker,
                report_date,
                account_nm,
                thstrm_amount
            FROM {table_name}
            WHERE ticker IS NOT NULL
              AND ticker != ''
            ORDER BY ticker, report_date
        """
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()


# ============================================================
# 3) 한 종목에 대한 전체 파이프라인
#    (raw df_ticker 만 받아서 처리, DB 접속 없음)
# ============================================================

def build_fs_df_from_raw(df: pd.DataFrame, ticker: str) -> Optional[pd.DataFrame]:
    """
    하나의 ticker에 대해:
    - IS/B/S/CF 정제
    - 누적 → 분기 보정
    - TTM, ROA, ROE, payout_ratio 계산
    최종 결과 DataFrame(date, 지표들, ticker) 반환
    """
    if df.empty:
        print(f"[WARN] {ticker}: 데이터 없음, skip")
        return None

    # 1) report_date 정리
    df["report_date"] = pd.to_datetime(df["report_date"])

    # 2) 손익계산서 pivot
    is_table = df.pivot_table(
        index="report_date",
        columns="account_nm",
        values="thstrm_amount",
        aggfunc="first"
    )

    is_table = is_table.sort_index()
    is_table = is_table.sort_index(axis=1)

    # -----------------------------
    # 2-1) 손익계산서 계정 정제
    # -----------------------------
    # 매출액_수정 = 매출액 or 수익(매출액) or (매출원가+매출총이익)
    매출액_s = safe_series(is_table, "매출액")
    매출액_s = 매출액_s.fillna(safe_series(is_table, "수익(매출액)"))
    if "매출원가" in is_table.columns and "매출총이익" in is_table.columns:
        매출액_s = 매출액_s.fillna(is_table["매출원가"] + is_table["매출총이익"])
    is_table["매출액_수정"] = 매출액_s

    # 영업이익_수정 = 영업이익 or 영업이익(손실)
    영업이익_s = safe_series(is_table, "영업이익")
    영업이익_s = 영업이익_s.fillna(safe_series(is_table, "영업이익(손실)"))
    is_table["영업이익_수정"] = 영업이익_s

    # 기타이익/손실
    기타이익_s = safe_series(is_table, "기타이익").fillna(
        safe_series(is_table, "기타수익")
    )
    기타손실_s = safe_series(is_table, "기타손실").fillna(
        safe_series(is_table, "기타비용")
    )
    is_table["기타이익_수정"] = 기타이익_s
    is_table["기타손실_수정"] = 기타손실_s

    # 법인세차감전순이익 / 법인세비용
    법차전_s = safe_series(is_table, "법인세비용차감전순이익").fillna(
        safe_series(is_table, "법인세비용차감전순이익(손실)")
    )
    법인세비용_s = safe_series(is_table, "법인세비용").fillna(
        safe_series(is_table, "법인세비용(수익)")
    )
    is_table["법인세차감전순이익_수정"] = 법차전_s
    is_table["법인세비용_수정"] = 법인세비용_s

    # 비지배주주 당기순이익
    비지배_s = safe_series(is_table, "비지배지분").fillna(
        safe_series(is_table, "비지배지분에 귀속되는 당기순이익(손실)")
    )
    is_table["당기순이익_비지배주주_수정"] = 비지배_s

    # 지배주주 당기순이익 결합
    cand_cols = [
        "분기순이익",
        "지배기업 소유주지분",
        "지배기업 소유지분",
        "지배기업의 소유주에게 귀속되는 당기순이익(손실)",
    ]
    avail = [c for c in cand_cols if c in is_table.columns]
    if avail:
        is_table["당기순이익_지배주주"] = is_table[avail].bfill(axis=1).iloc[:, 0]
    else:
        is_table["당기순이익_지배주주"] = pd.Series(
            index=is_table.index, dtype="float64"
        )

    # 총 당기순이익 = 지배 + 비지배
    is_table["당기순이익"] = (
        is_table["당기순이익_지배주주"].fillna(0)
        + is_table["당기순이익_비지배주주_수정"].fillna(0)
    )

    # -----------------------------
    # 2-2) IS 누적 → 분기 보정
    # -----------------------------
    value_cols_is = [
        "매출액_수정",
        "매출원가",
        "매출총이익",
        "판매비와관리비",
        "영업이익_수정",
        "금융비용",
        "금융수익",
        "기타손익_수정",
        "법인세차감전순이익_수정",
        "법인세비용_수정",
        "당기순이익_지배주주",
        "당기순이익_비지배주주_수정",
        "당기순이익",
    ]
    value_cols_is = [c for c in value_cols_is if c in is_table.columns]

    is_table_adj = adjust_quarterly_from_index(is_table, value_cols_is)

    # 법인세차감전순이익 - 법인세비용 = 당기순이익_수정
    if (
        "법인세차감전순이익_수정" in is_table_adj.columns
        and "법인세비용_수정" in is_table_adj.columns
    ):
        is_table_adj["당기순이익_수정"] = (
            is_table_adj["법인세차감전순이익_수정"]
            - is_table_adj["법인세비용_수정"]
        )

    # -----------------------------
    # 3) 대차대조표 (자본총계) / 현금흐름표 정제
    # -----------------------------
    # 자본총계 = 자산총계 - 부채총계
    if "자산총계" in is_table.columns and "부채총계" in is_table.columns:
        is_table["자본총계_수정"] = is_table["자산총계"] - is_table["부채총계"]

    bs_keep = [
        "자산총계",
        "유동자산",
        "재고자산",
        "유형자산",
        "부채총계",
        "유동부채",
        "비유동부채",
        "자본총계_수정",
    ]
    bs_keep = [c for c in bs_keep if c in is_table.columns]
    bs_table = is_table[bs_keep]

    # 배당금의지급 계정 통합
    배당_s = safe_series(is_table, "배당금의지급")
    배당_s = 배당_s.fillna(safe_series(is_table, "배당금 지급"))
    배당_s = 배당_s.fillna(safe_series(is_table, "배당금의 지급"))
    is_table["배당금의지급"] = 배당_s
    is_table["배당금의지급_수정"] = 배당_s

    # 현금흐름 계정
    영업CF_s = safe_series(is_table, "영업활동 현금흐름").fillna(
        safe_series(is_table, "영업활동현금흐름")
    )
    투자CF_s = safe_series(is_table, "투자활동 현금흐름").fillna(
        safe_series(is_table, "투자활동현금흐름")
    )
    재무CF_s = safe_series(is_table, "재무활동 현금흐름").fillna(
        safe_series(is_table, "재무활동현금흐름")
    )

    is_table["영업활동현금흐름_수정"] = 영업CF_s
    is_table["투자활동현금흐름_수정"] = 투자CF_s
    is_table["재무활동현금흐름_수정"] = 재무CF_s

    cf_keep = [
        "영업활동현금흐름_수정",
        "투자활동현금흐름_수정",
        "재무활동현금흐름_수정",
        "배당금의지급_수정",
        "법인세 납부액",
    ]
    cf_keep = [c for c in cf_keep if c in is_table.columns]
    cf_table = is_table[cf_keep]

    # 현금흐름 누적 → 분기 보정
    cf_value_cols = cf_keep[:]  # 모두 누적값이라고 가정
    cf_table = cumulative_to_quarterly(cf_table, cf_value_cols)

    # -----------------------------
    # 4) IS / BS / CF 결합
    # -----------------------------
    target_cols = [
        "매출액_수정",
        "매출원가",
        "매출총이익",
        "판매비와관리비",
        "영업이익_수정",
        "기타이익_수정",
        "기타손실_수정",
        "법인세차감전순이익_수정",
        "법인세비용_수정",
        "당기순이익_수정",
    ]
    target_cols = [c for c in target_cols if c in is_table_adj.columns]
    is_table_ = is_table_adj[target_cols]

    fs_df = pd.concat([is_table_, bs_table, cf_table], axis=1)

    # "_수정" suffix 제거
    fs_df = fs_df.rename(
        columns={
            c: c.replace("_수정", "")
            for c in fs_df.columns
            if c.endswith("_수정")
        }
    )

    # -----------------------------
    # 5) TTM / ROA·ROE / payout 계산
    # -----------------------------
    required_cols = [
        "매출액",
        "매출총이익",
        "영업이익",
        "당기순이익",
        "자산총계",
        "자본총계",
        "배당금의지급",
    ]
    missing = [c for c in required_cols if c not in fs_df.columns]
    if missing:
        print(f"[WARN] {ticker}: 비율 계산에 필요한 칼럼 부족 -> {missing}, skip")
        return None

    fs_df.index.name = "date"
    fs_df.index = pd.to_datetime(fs_df.index)
    fs_df = fs_df.sort_index()

    # TTM
    fs_df["매출총이익_ttm"] = fs_df["매출총이익"].rolling(4).sum()
    fs_df["당기순이익_ttm"] = fs_df["당기순이익"].rolling(4).sum()
    fs_df["영업이익_ttm"] = fs_df["영업이익"].rolling(4).sum()
    fs_df["매출액_ttm"] = fs_df["매출액"].rolling(4).sum()

    # 전년동기 자산/자본
    fs_df["자본총계_lag4"] = fs_df["자본총계"].shift(4)
    fs_df["자산총계_lag4"] = fs_df["자산총계"].shift(4)

    fs_df["자본총계_평균"] = (fs_df["자본총계"] + fs_df["자본총계_lag4"]) / 2
    fs_df["자산총계_평균"] = (fs_df["자산총계"] + fs_df["자산총계_lag4"]) / 2

    # 비율
    fs_df["GPM_ttm"] = safe_divide(fs_df["매출총이익_ttm"], fs_df["매출액_ttm"])
    fs_df["OPM_ttm"] = safe_divide(fs_df["영업이익_ttm"], fs_df["매출액_ttm"])
    fs_df["NIM_ttm"] = safe_divide(fs_df["당기순이익_ttm"], fs_df["매출액_ttm"])

    fs_df["ROA"] = safe_divide(fs_df["당기순이익_ttm"], fs_df["자산총계_평균"])
    fs_df["ROE"] = safe_divide(fs_df["당기순이익_ttm"], fs_df["자본총계_평균"])

    fs_df["payout_ratio"] = safe_divide(
        fs_df["배당금의지급"], fs_df["당기순이익_ttm"]
    )

    # ticker 붙이고 date 컬럼으로 reset
    fs_df = fs_df.assign(ticker=ticker).reset_index()

    return fs_df


# ============================================================
# 4) 전체 ticker 일괄 처리
# ============================================================

# 1) 전체 raw 데이터 로드
all_raw_df = fetch_all_fs_data(db_info, TABLE_NAME)
print("[INFO] raw shape:", all_raw_df.shape)

# 2) ticker 단위 groupby
grouped = all_raw_df.groupby("ticker")

all_list = []

# 필요하면 테스트용으로 20개만 먼저 돌려보고 싶으면:
# for i, (tkr, df_t) in enumerate(grouped):
#     if i >= 20:
#         break
#     ...

for tkr, df_t in grouped:
    print(f"[INFO] processing ticker {tkr} ...")
    try:
        one_df = build_fs_df_from_raw(df_t, tkr)
        if one_df is not None and not one_df.empty:
            all_list.append(one_df)
    except Exception as e:
        print(f"[ERROR] {tkr}: {e}")

if all_list:
    all_fs_df = pd.concat(all_list, ignore_index=True)
    print("[INFO] 최종 DataFrame shape:", all_fs_df.shape)
else:
    all_fs_df = pd.DataFrame()
    print("[WARN] 유효한 결과가 없습니다.")



[INFO] raw shape: (1264220, 5)
[INFO] processing ticker 000070 ...
[INFO] processing ticker 000080 ...
[INFO] processing ticker 000100 ...
[INFO] processing ticker 000120 ...
[INFO] processing ticker 000150 ...
[INFO] processing ticker 000210 ...
[INFO] processing ticker 000240 ...
[INFO] processing ticker 000250 ...
[INFO] processing ticker 000270 ...
[INFO] processing ticker 000500 ...
[INFO] processing ticker 000640 ...
[INFO] processing ticker 000660 ...
[INFO] processing ticker 000670 ...
[INFO] processing ticker 000720 ...
[INFO] processing ticker 000810 ...
[WARN] 000810: 비율 계산에 필요한 칼럼 부족 -> ['매출총이익'], skip
[INFO] processing ticker 000880 ...
[INFO] processing ticker 000990 ...
[INFO] processing ticker 001040 ...
[INFO] processing ticker 001120 ...
[INFO] processing ticker 001430 ...
[INFO] processing ticker 001440 ...
[INFO] processing ticker 001450 ...
[WARN] 001450: 비율 계산에 필요한 칼럼 부족 -> ['매출총이익'], skip
[INFO] processing ticker 001680 ...
[INFO] processing ticker 001720 ...
[WA

In [353]:
# 1) 손익계산서(IS)만 필터링
is_df = df[df["sj_div"] == "IS"].copy()

# 2) 당기순이익 관련 키워드(한글/영문)
net_income_keywords = [
    "당기순이익", "순이익", "이익(손실)",
    "ProfitLoss", "NetIncome",
    "Profit(Loss)", "ProfitLossAttributable"
]

# 3) 해당 키워드 포함 account_nm 또는 account_id 추출
mask_net_income = False
for kw in net_income_keywords:
    mask_net_income |= (
        is_df["account_nm"].astype(str).str.contains(kw, na=False) |
        is_df["account_id"].astype(str).str.contains(kw, na=False)
    )

net_income_candidates = (
    is_df.loc[mask_net_income, ["account_id", "account_nm"]]
        .drop_duplicates()
        .sort_values("account_id")
)

net_income_accounts = net_income_candidates["account_id"].tolist()

continuing_keywords = ["계속", "continuing", "Continuing"]

mask_cont = False
for kw in continuing_keywords:
    mask_cont |= (
        is_df["account_nm"].astype(str).str.contains(kw, na=False) |
        is_df["account_id"].astype(str).str.contains(kw, na=False)
    )

continuing_candidates = (
    is_df.loc[mask_cont, ["account_id", "account_nm"]]
        .drop_duplicates()
        .sort_values("account_id")
)

continuing_accounts = continuing_candidates["account_id"].tolist()

print("📌 계속사업 관련 account_id:")
for x in continuing_accounts:
    print("    ", repr(x))

account_groups = {

    # -------------------------------------------------
    # 1) 매출액 (Revenue)
    # -------------------------------------------------
    "revenue": [
        "ifrs_Revenue",
        "ifrs-full_Revenue",
    ],

    # -------------------------------------------------
    # 2) 매출총이익 (Gross Profit)
    # -------------------------------------------------
    "gross_profit": [
        "ifrs_GrossProfit",
        "ifrs-full_GrossProfit",
    ],

    # -------------------------------------------------
    # 3) 영업이익 (Operating Income)
    # -------------------------------------------------
    "operating_income": [
        "dart_OperatingIncomeLoss",
    ],

    # -------------------------------------------------
    # 4) 당기순이익 (Net Income) - 총당기순이익
    #     손익계산서 기준 전체 당기순이익 + CF용 당기순이익 포함
    # -------------------------------------------------
    "net_income_total": [
        "ifrs_ProfitLoss",
        "ifrs-full_ProfitLoss",
        "dart_ProfitLossForStatementOfCashFlows",
    ],

    # -------------------------------------------------
    # 4-1) 계속사업 관련 손익 (계속영업이익, 계속사업 법인세 등)
    # -------------------------------------------------
    "continuing_operations": [
        "ifrs_ProfitLossBeforeTax",
        "ifrs-full_ProfitLossBeforeTax",
    ],

    # -------------------------------------------------
    # 4-2) 계속사업 법인세 등
    # -------------------------------------------------
    "income_tax": [
        "ifrs_IncomeTaxExpenseContinuingOperations",             # 계속사업 법인세비용
        "ifrs-full_IncomeTaxExpenseContinuingOperations",        # (full IFRS) 계속사업 법인세비용
    ],


    # -------------------------------------------------
    # 5) 당기순이익 - 지배기업 소유주 귀속
    # -------------------------------------------------
    "net_income_parent": [
        "ifrs_ProfitLossAttributableToOwnersOfParent",
        "ifrs-full_ProfitLossAttributableToOwnersOfParent",
    ],

    # -------------------------------------------------
    # 6) 당기순이익 - 비지배지분 귀속
    # -------------------------------------------------
    "net_income_nci": [
        "ifrs_ProfitLossAttributableToNoncontrollingInterests",
        "ifrs-full_ProfitLossAttributableToNoncontrollingInterests",
    ],

    "discontinued_pl_accounts" : ["ifrs_ProfitLossFromDiscontinuedOperations"],
}

account_groups_bs = {

    # -------------------------------------------------
    # 1) 자산총계 (Total Assets)
    # -------------------------------------------------
    "assets_total": [
        "ifrs_Assets",              # IFRS 전체 자산
        "ifrs-full_Assets",         # IFRS full
    ],

    # -------------------------------------------------
    # 2) 유동자산 (Current Assets)
    # -------------------------------------------------
    "current_assets": [
        "ifrs_CurrentAssets",
        "ifrs-full_CurrentAssets",
    ],

    # -------------------------------------------------
    # 3) 재고자산 (Inventories)
    # -------------------------------------------------
    "inventories": [
        "ifrs_Inventories",
        "ifrs-full_Inventories",
    ],

    # -------------------------------------------------
    # 4) 유형자산 (Property, Plant and Equipment)
    # -------------------------------------------------
    "ppe": [
        "ifrs_PropertyPlantAndEquipment",
        "ifrs-full_PropertyPlantAndEquipment",
    ],

    # -------------------------------------------------
    # 5) 부채총계 (Total Liabilities)
    # -------------------------------------------------
    "liabilities_total": [
        "ifrs_Liabilities",
        "ifrs-full_Liabilities",
    ],

    # -------------------------------------------------
    # 6) 유동부채 (Current Liabilities)
    # -------------------------------------------------
    "current_liabilities": [
        "ifrs_CurrentLiabilities",
        "ifrs-full_CurrentLiabilities",
    ],

    # -------------------------------------------------
    # 7) 비유동부채 (Noncurrent Liabilities)
    # -------------------------------------------------
    "noncurrent_liabilities": [
        "ifrs_NoncurrentLiabilities",
        "ifrs-full_NoncurrentLiabilities",
    ],
}

account_groups_cf = {

    # -------------------------------------------------
    # 1) 영업활동 현금흐름
    # -------------------------------------------------
    "cf_operating": [
        "ifrs_CashFlowsFromUsedInOperatingActivities",
        "ifrs-full_CashFlowsFromUsedInOperatingActivities",
    ],

    # 영업활동 조정항목 (당기순이익 → 영업CF로 reconcile)
    "cf_adjustments": [
        "ifrs_AdjustmentsForReconcileProfitLoss",
        "ifrs-full_CashFlowsFromUsedInOperations",  # Operations 기준도 함께
    ],

    # -------------------------------------------------
    # 2) 투자활동 현금흐름
    # -------------------------------------------------
    "cf_investing": [
        "ifrs_CashFlowsFromUsedInInvestingActivities",
        "ifrs-full_CashFlowsFromUsedInInvestingActivities",
    ],

    # -------------------------------------------------
    # 3) 재무활동 현금흐름
    # -------------------------------------------------
    "cf_financing": [
        "ifrs_CashFlowsFromUsedInFinancingActivities",
        "ifrs-full_CashFlowsFromUsedInFinancingActivities",
    ],

    # -------------------------------------------------
    # 4) 법인세 납부(환급) - 영업활동 분류
    # -------------------------------------------------
    "cf_tax_operating": [
        "ifrs_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
        "ifrs-full_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 5) 이자수익/이자비용 - 영업활동 분류
    # -------------------------------------------------
    "cf_interest_received": [
        "ifrs_InterestReceivedClassifiedAsOperatingActivities",
        "ifrs-full_InterestReceivedClassifiedAsOperatingActivities",
    ],
    "cf_interest_paid": [
        "ifrs_InterestPaidClassifiedAsOperatingActivities",
        "ifrs-full_InterestPaidClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 6) 배당수익/배당금 지급
    # -------------------------------------------------
    "cf_dividends_received": [
        "ifrs_DividendsReceivedClassifiedAsOperatingActivities",
        "ifrs-full_DividendsReceivedClassifiedAsOperatingActivities",
    ],
    "cf_dividends_paid": [
        "ifrs_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs_DividendsPaid",
        "ifrs-full_DividendsPaid",
    ],

    # -------------------------------------------------
    # 7) 차입/상환 (재무활동 디테일)
    # -------------------------------------------------
    "cf_borrowings": [
        "ifrs_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
    ],
    "cf_repayments": [
        "ifrs_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
    ],

    # -------------------------------------------------
    # 8) 기초/기말 현금 및 현금성자산, 증감, 환율효과
    # -------------------------------------------------
    "cf_beginning_cash": [
        "dart_CashAndCashEquivalentsAtBeginningOfPeriodCf",
    ],
    "cf_ending_cash": [
        "dart_CashAndCashEquivalentsAtEndOfPeriodCf",
    ],
    "cf_increase_decrease_cash": [
        "ifrs_IncreaseDecreaseInCashAndCashEquivalents",
        "ifrs-full_IncreaseDecreaseInCashAndCashEquivalents",
    ],
    "cf_fx_effect": [
        "ifrs_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
        "ifrs-full_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
    ],

    # -------------------------------------------------
    # 9) 기타: 단기예금/투자, 리스상환, 정부보조금 등
    #    (필요시 나중에 세분화해서 쓰실 수 있게 모아둠)
    # -------------------------------------------------
    "cf_other_investing": [
        "ifrs-full_ProceedsFromSalesOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromSalesOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromGovernmentGrantsClassifiedAsInvestingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsInvestingActivities",
    ],
    "cf_other_financing": [
        "ifrs-full_PaymentsOfFinanceLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_PaymentsOfLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsFinancingActivities",
        "ifrs-full_SaleOrIssueOfTreasuryShares",
        "ifrs-full_IncreaseDecreaseThroughSharebasedPaymentTransactions",
    ],
}

def extract_is_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups: dict) -> pd.DataFrame:
    """
    손익계산서(IS)에서 account_groups[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = ((df["sj_div"] == "IS") | (df["sj_div"] == "CIS")) & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out


def extract_bs_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_bs: dict) -> pd.DataFrame:

    ids = account_groups_bs.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "BS") & df["account_id"].isin(ids)
    return df.loc[mask].copy()

def extract_cf_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_cf: dict) -> pd.DataFrame:
    """
    현금흐름표(CF)에서 account_groups_cf[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups_cf.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "CF") & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out

def make_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    입력된 현금흐름표(DataFrame)에서
    - account_id별 가장 많이 출현한 account_nm을 대표 계정명으로 선택하여
    - report_date를 index로 하고
    - 대표 account_nm을 column으로 하는
    pivot table을 생성하는 함수.
    """
    if df.empty:
        raise ValueError("입력된 DataFrame이 비어 있습니다.")

    df = df.copy()

    # 1) report_date datetime 변환
    df["report_date"] = pd.to_datetime(df["report_date"])

    # 2) account_id별 대표 account_nm 계산
    rep_name_map = (
        df.groupby("account_id")["account_nm"]
          .agg(lambda s: s.value_counts().idxmax())
          .to_dict()
    )

    # 3) 대표 계정명 컬럼 생성
    df["account_nm_rep"] = df["account_id"].map(rep_name_map)

    # 4) pivot table 생성
    pivot_df = df.pivot_table(
        index="report_date",
        columns="account_nm_rep",
        values="thstrm_amount",
        aggfunc="first"    # 필요시 sum, mean 등으로 변경 가능
    ).sort_index()

    # 5) 컬럼을 이름순으로 정렬
    pivot_df = pivot_df.reindex(sorted(pivot_df.columns), axis=1)

    return pivot_df


def adjust_quarterly_q4_only(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """
    1Q, 2Q, 3Q는 원래 값 그대로 두고,
    4Q(12월)만 `FY - (Q1+Q2+Q3)`로 조정하는 함수.

    전제:
      - df.index: DatetimeIndex (분기말 날짜)
      - value_cols: 조정할 수치 칼럼 리스트
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)

    # 연도별로 처리
    for year in sorted(out.index.year.unique()):
        mask_year = out.index.year == year
        sub = out.loc[mask_year].sort_index()

        q1_idx = sub.index[sub.index.month == 3]
        q2_idx = sub.index[sub.index.month == 6]
        q3_idx = sub.index[sub.index.month == 9]
        q4_idx = sub.index[sub.index.month == 12]

        # 4분기가 없으면 그 해는 스킵
        if len(q4_idx) == 0:
            continue

        # 4분기 행(여러 개라면 첫 번째만 사용한다고 가정)
        q4_i = q4_idx[0]

        for col in value_cols:
            if col not in out.columns:
                continue

            fy = out.loc[q4_i, col]
            if pd.isna(fy):
                continue

            prev_sum = (
                out.loc[q1_idx, col].fillna(0).sum()
                + out.loc[q2_idx, col].fillna(0).sum()
                + out.loc[q3_idx, col].fillna(0).sum()
            )

            # ✅ 오직 4분기 값만 조정
            out.loc[q4_i, col] = fy - prev_sum

    return out

📌 계속사업 관련 account_id:
     'ifrs-full_IncomeTaxExpenseContinuingOperations'
     'ifrs-full_IncomeTaxExpenseContinuingOperations'
     'ifrs-full_ProfitLossFromContinuingOperations'
     'ifrs_IncomeTaxExpenseContinuingOperations'
     'ifrs_ProfitLossFromContinuingOperations'


In [371]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

df = fetch_fs_data_by_ticker(db_info, "000120")

if df.empty:
    print("데이터 없음")
else:
    display(df.head())

# 1) report_date 를 datetime 으로 변환 (안 되어 있으면)
df["report_date"] = pd.to_datetime(df["report_date"])

# 1) 매출액 관련 계정만 필터링

#
ni_total_df = extract_is_group(df, "net_income_total", account_groups)
# 2) 계속사업 관련 손익 추출
cont_ops_df = extract_is_group(df, "continuing_operations", account_groups)
# 3) 지배주주 귀속 순이익
ni_parent_df = extract_is_group(df, "net_income_parent", account_groups)
# 4) 비지배지분 귀속 순이익
ni_nci_df = extract_is_group(df, "net_income_nci", account_groups)
# 5) 법인세 비용
tax_df = extract_is_group(df, "income_tax", account_groups)
disc_df = extract_is_group(df, "discontinued_pl_accounts", account_groups)
rev_df = df[df["account_id"].isin(account_groups["revenue"])]
gp_df = df[df["account_id"].isin(account_groups["gross_profit"])]
op_df = df[df["account_id"].isin(account_groups["operating_income"])]
# tax_df = df[df["account_id"].isin(account_groups["net_income_total"])]
# ni_parent = df[df["account_id"].isin(account_groups["net_income_parent"])]
# ni_nci = df[df["account_id"].isin(account_groups["net_income_nci"])]


# 2) 대차대조표 계정만 필터링
assets_df = extract_bs_group(df, "assets_total", account_groups_bs)
current_assets_df = extract_bs_group(df, "current_assets", account_groups_bs)
inventories_df = extract_bs_group(df, "inventories", account_groups_bs)
ppe_df = extract_bs_group(df, "ppe", account_groups_bs)
liabilities_df = extract_bs_group(df, "liabilities_total", account_groups_bs)
current_liab_df = extract_bs_group(df, "current_liabilities", account_groups_bs)
noncurrent_liab_df = extract_bs_group(df, "noncurrent_liabilities", account_groups_bs)

# 3) 현금흐름표 계정만 필터링

cf_op_df   = extract_cf_group(df, "cf_operating", account_groups_cf)
cf_inv_df  = extract_cf_group(df, "cf_investing", account_groups_cf)
cf_fin_df  = extract_cf_group(df, "cf_financing", account_groups_cf)
cf_tax_df  = extract_cf_group(df, "cf_tax_operating", account_groups_cf)
cf_div_df  = extract_cf_group(df, "cf_dividends_paid", account_groups_cf)

rev_pivot = make_pivot(rev_df)
gp_pivot = make_pivot(gp_df)
op_pivot = make_pivot(op_df)
cont_ops_pivot = make_pivot(cont_ops_df)
tax_pivot = make_pivot(tax_df)
# # 2) pivot table 생성
# is_table = df.pivot_table(
#     index="report_date",
#     columns="account_nm",
#     values="thstrm_amount",
#     aggfunc="first"   # 같은 날짜+계정명 중복이 있을 경우 첫 값을 사용
# )
#
# # 3) 컬럼/인덱스 정리
# is_table = is_table.sort_index()         # 날짜순 정렬
# is_table = is_table.sort_index(axis=1)   # 계정명 알파벳 순(원하면 제거)
#
# try:
#     is_table['매출액_수정'] = is_table['매출액'].fillna(is_table["수익(매출액)"])
# except:
#     pass
# # is_table['매출액_수정'] = is_table['매출액']
# is_table['영업이익_수정'] = is_table["영업이익"].fillna(is_table["영업이익(손실)"])
# # is_table['기타이익_수정'] = is_table["기타이익"].fillna(is_table["기타수익"])
# # is_table['기타손실_수정'] = is_table["기타손실"].fillna(is_table["기타비용"])
# is_table['법인세차감전순이익_수정'] = is_table["법인세비용차감전순이익"].fillna(is_table["법인세비용차감전순이익(손실)"])
# is_table['법인세비용_수정'] = is_table["법인세비용"].fillna(is_table["법인세비용(수익)"])
# is_table['당기순이익_비지배주주_수정'] = is_table["비지배지분"].fillna(is_table["비지배지분에 귀속되는 당기순이익(손실)"])
# # 매출액 최종 수정
# is_table["매출액_수정"] = is_table["매출액_수정"].fillna(is_table["매출원가"] + is_table["매출총이익"])
#
# # 결합 대상 컬럼 목록
# cols = [
#     '분기순이익',
#     '지배기업 소유주지분',
#     '지배기업 소유지분',
#     '지배기업의 소유주에게 귀속되는 당기순이익(손실)'
# ]
#
# # 존재하는 컬럼만 필터링
# available_cols = [c for c in cols if c in is_table.columns]
#
# if not available_cols:
#     print("[WARN] 결합할 칼럼이 없습니다.")
# else:
#     # row-wise 로 결합 → 첫 번째로 NaN이 아닌 값을 선택
#     is_table['당기순이익_지배주주'] = is_table[available_cols].bfill(axis=1).iloc[:, 0]
#
# is_table['당기순이익'] = is_table['당기순이익_지배주주'] + is_table['당기순이익_비지배주주_수정']
#
# value_cols = [
#     "매출액_수정",
#     "매출원가",
#     "매출총이익",
#     "판매비와관리비",
#     "영업이익_수정",
#     "금융비용",
#     "금융수익",
#     "기타손익_수정",
#     "법인세차감전순이익_수정",
#     "법인세비용_수정",
#     "당기순이익_지배주주",
#     "당기순이익_비지배주주_수정",
#     "당기순이익"
# ]
#
# is_table_adj = adjust_quarterly_from_index(is_table, value_cols)
#
# is_table_adj["당기순이익_수정"] = (
#     is_table_adj["법인세차감전순이익_수정"]
#     - is_table_adj["법인세비용_수정"]
# )
#
#
# value_cols = [
#     "당기순이익_지배주주"
# ]
#
# is_table_adj = cumulative_to_quarterly(is_table_adj, value_cols)
#
# # ============================================================
# # 대차대조표 계정 조정
# # ============================================================
#
# is_table['자본총계_수정'] = is_table["자산총계"] - is_table["부채총계"]
# bs_table = is_table[['자산총계', '유동자산', '재고자산', '유형자산', '부채총계', '유동부채', '비유동부채', '자본총계_수정']]
#
# #    "배당금 지급" 또는 "배당금의 지급" 값 중에서 채움
# is_table["배당금의지급"] = (is_table["배당금의지급"].fillna(is_table["배당금 지급"]).fillna(is_table["배당금의 지급"]))
#
# #    "배당금 지급" 또는 "배당금의 지급" 값 중에서 채움
# is_table["배당금의지급_수정"] = (is_table["배당금의지급"].fillna(is_table["배당금 지급"]).fillna(is_table["배당금의 지급"]))
# # is_table[['법인세 납부액', '영업활동현금흐름','투자활동현금흐름' ]]
# is_table["영업활동현금흐름_수정"] =  is_table["영업활동 현금흐름"].fillna(is_table["영업활동현금흐름"])
# is_table["투자활동현금흐름_수정"] =  is_table["투자활동 현금흐름"].fillna(is_table["투자활동현금흐름"])
# is_table["재무활동현금흐름_수정"] =  is_table["재무활동 현금흐름"].fillna(is_table["재무활동현금흐름"])
#
# cf_table = is_table[["영업활동현금흐름_수정", "투자활동현금흐름_수정", "재무활동현금흐름_수정", "배당금의지급_수정", '법인세 납부액']]
#
# value_cols = [
#     "영업활동현금흐름_수정",
#     "투자활동현금흐름_수정",
#     "재무활동현금흐름_수정",
#     "배당금의지급_수정",
#     "법인세 납부액"
# ]
#
# cf_table = cumulative_to_quarterly(cf_table, value_cols)
#
# target_cols = ['매출액_수정', '매출원가', '매출총이익', '판매비와관리비','영업이익_수정',
#                '기타이익_수정', '기타손실_수정', '법인세차감전순이익_수정', '법인세비용_수정', '당기순이익_수정']
#
# is_table_ = is_table_adj[target_cols]
#
# fs_df = pd.concat([is_table_, bs_table, cf_table], axis=1)
#
# fs_df = fs_df.rename(
#     columns={col: col.replace("_수정", "") for col in fs_df.columns if col.endswith("_수정")}
# )
#
# # fs_df 가 아래 칼럼들을 모두 가지고 있다고 가정합니다.
# # ['매출액','매출총이익','영업이익','당기순이익','자산총계','자본총계','배당금의지급']
#
# required_cols = [
#     '매출액',
#     '매출총이익',
#     '영업이익',
#     '당기순이익',
#     '자산총계',
#     '자본총계',
#     '배당금의지급'
# ]
#
# missing = [c for c in required_cols if c not in fs_df.columns]
# if missing:
#     raise KeyError(f"다음 칼럼이 없습니다: {missing}")
#
#
# fs_df.index.name = 'date'
#
# # 2) 인덱스가 datetime 이 아니면 변환
# fs_df.index = pd.to_datetime(fs_df.index)
#
# # 3) 날짜 기준 정렬
# fs_df = fs_df.sort_index()
#
# fs_df['매출총이익_ttm'] = fs_df['매출총이익'].rolling(4).sum()
# fs_df['당기순이익_ttm'] = fs_df['당기순이익'].rolling(4).sum()
# fs_df['영업이익_ttm']   = fs_df['영업이익'].rolling(4).sum()
# fs_df['매출액_ttm']     = fs_df['매출액'].rolling(4).sum()
#
# # 1) 전년 동기대비(4분기 전) 자본총계
# fs_df['자본총계_lag4'] = fs_df['자본총계'].shift(4)
# fs_df['자산총계_lag4'] = fs_df['자산총계'].shift(4)
#
# # 2) 자본총계 평균 (기초+기말)/2 개념
# fs_df['자본총계_평균'] = (fs_df['자본총계'] + fs_df['자본총계_lag4']) / 2
# fs_df['자산총계_평균'] = (fs_df['자산총계'] + fs_df['자산총계_lag4']) / 2
#
# # 1) GPM: Gross Profit Margin
# fs_df['GPM_ttm'] = safe_divide(fs_df['매출총이익_ttm'], fs_df['매출액_ttm'])
#
# # 2) OPM: Operating Profit Margin
# fs_df['OPM_ttm'] = safe_divide(fs_df['영업이익_ttm'], fs_df['매출액_ttm'])
#
# # 3) NIM: Net Income Margin
# fs_df['NIM_ttm'] = safe_divide(fs_df['당기순이익_ttm'], fs_df['매출액_ttm'])
#
#
# # 1) GPM: Gross Profit Margin
# fs_df['GPM_ttm'] = safe_divide(fs_df['매출총이익_ttm'], fs_df['매출액_ttm'])
#
# # 2) OPM: Operating Profit Margin
# fs_df['OPM_ttm'] = safe_divide(fs_df['영업이익_ttm'], fs_df['매출액_ttm'])
#
# # 3) NIM: Net Income Margin
# fs_df['NIM_ttm'] = safe_divide(fs_df['당기순이익_ttm'], fs_df['매출액_ttm'])
#
# # 4) ROA: Return on Assets
# fs_df['ROA'] = safe_divide(fs_df['당기순이익_ttm'], fs_df['자산총계_평균'])
#
# # 5) ROE: Return on Equity
# fs_df['ROE'] = safe_divide(fs_df['당기순이익_ttm'], fs_df['자본총계_평균'])
#
# # 6) 배당성향: payout_ratio
# fs_df['payout_ratio'] = safe_divide(fs_df['배당금의지급'], fs_df['당기순이익_ttm'])

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
0,00113410,2015,11011,FY,ifrs_OtherCurrentNonfinancialLiabilities,BS,재무상태표,기타금융부채,제 106 기,1.474042e+11,2015-12-31,000120
1,00113410,2015,11011,FY,ifrs_OtherNoncurrentFinancialLiabilities,BS,재무상태표,기타금융부채,제 106 기,5.283331e+09,2015-12-31,000120
2,00113410,2015,11011,FY,ifrs_OtherNoncurrentNonfinancialAssets,BS,재무상태표,기타금융자산,제 106 기,1.024911e+10,2015-12-31,000120
3,00113410,2015,11011,FY,ifrs_OtherNoncurrentNonfinancialLiabilities,BS,재무상태표,기타부채,제 106 기,2.759200e+09,2015-12-31,000120
4,00113410,2015,11011,FY,dart_LongTermTradeAndOtherNonCurrentReceivable...,BS,재무상태표,기타수취채권,제 106 기,1.133509e+11,2015-12-31,000120


In [372]:
rev_pivot = make_pivot(rev_df)
gp_pivot = make_pivot(gp_df)
op_pivot = make_pivot(op_df)
cont_ops_pivot = make_pivot(cont_ops_df)
tax_pivot = make_pivot(tax_df)

is_table = pd.concat([rev_pivot, gp_pivot, op_pivot, cont_ops_pivot, tax_pivot], axis=1)
is_table.index.name = 'report_date'

value_cols = is_table.columns.tolist()
is_table_adj = adjust_quarterly_q4_only(is_table, value_cols)
# 계정 일치를 위해 신규 칼럼 이름 부여
is_table_adj.columns = ['매출액', '매출총이익', '영업이익', '법인세비용차감전순이익', '법인세비용']
is_table_adj['당기순이익'] = is_table_adj['법인세비용차감전순이익'] - is_table_adj['법인세비용']

In [373]:
is_table_adj

,매출액,매출총이익,영업이익,법인세비용차감전순이익,법인세비용,당기순이익
report_date,,,,,,
2015-12-31,5.055766e+12,5.492598e+11,1.866337e+11,8.031382e+10,1.709636e+10,6.321746e+10
2016-03-31,1.445216e+12,1.587311e+11,5.322637e+10,2.852063e+10,2.200019e+09,2.632061e+10
2016-06-30,1.513557e+12,1.704746e+11,5.966426e+10,3.712287e+10,8.675811e+09,2.844706e+10
2016-09-30,1.490306e+12,1.718308e+11,6.031498e+10,3.836138e+09,3.282465e+09,5.536731e+08
2016-12-31,1.632867e+12,1.700829e+11,5.523888e+10,2.153977e+10,8.650987e+09,1.288879e+10
2017-03-31,1.594923e+12,1.730115e+11,5.113910e+10,1.293340e+10,8.500382e+09,4.433013e+09
2017-06-30,1.707834e+12,1.858287e+11,6.185405e+10,3.791897e+10,1.339394e+10,2.452504e+10
2017-09-30,1.873206e+12,1.997162e+11,6.262631e+10,2.040362e+10,9.597982e+09,1.080564e+10
2017-12-31,1.934428e+12,1.975992e+11,6.003197e+10,1.880231e+09,2.756074e+09,-8.758427e+08


In [350]:
is_table.tail(10)

account_nm_rep,수익(매출액),매출총이익,영업이익,법인세비용차감전순이익(손실),법인세비용
report_date,,,,,
2023-06-30,6.000553e+13,1.835834e+13,6.685470e+11,1.712995e+12,-1.057600e+10
2023-09-30,6.740465e+13,2.078593e+13,2.433534e+12,3.942601e+12,-1.901570e+12
2023-12-31,2.589355e+14,7.854691e+13,6.566976e+12,1.100626e+13,-4.480835e+12
2024-03-31,7.191560e+13,2.602927e+13,6.606009e+12,7.706723e+12,9.520150e+11
2024-06-30,7.406830e+13,2.975628e+13,1.044388e+13,1.159534e+13,1.753999e+12
2024-09-30,7.909873e+13,3.000365e+13,9.183371e+12,1.032041e+13,2.195080e+11
2024-12-31,3.008709e+14,1.143086e+14,3.272596e+13,3.752973e+13,3.078383e+12
2025-03-31,7.914050e+13,2.813057e+13,6.685272e+12,9.151576e+12,9.286980e+11
2025-06-30,7.456632e+13,2.549675e+13,4.676057e+12,5.756129e+12,6.396940e+11


In [352]:
test = pd.DataFrame(
    {
        "매출액": [100, 300, 600, 1000],  # Q1, Q2, Q3, FY(누적)
    },
    index=pd.to_datetime(["2024-03-31", "2024-06-30", "2024-09-30", "2024-12-31"])
)
test.index.name = "report_date"

adj = adjust_quarterly_from_index(test, ["매출액"])
print(adj)

               매출액
report_date       
2024-03-31   100.0
2024-06-30   200.0
2024-09-30   300.0
2024-12-31   400.0


In [283]:
df[df['sj_div'] == 'IS']

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [297]:
df[(df['bsns_year'] == 2025)]

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
3650,00145109,2025,11012,H1,ifrs-full_InvestmentAccountedForUsingEquityMethod,BS,재무상태표,관계기업 및 공동기업투자,제 103 기 반기말,4.271225e+11,2025-06-30,000100
3651,00145109,2025,11012,H1,ifrs-full_AdditionalPaidinCapital,BS,재무상태표,기타불입자본,제 103 기 반기말,-6.732254e+10,2025-06-30,000100
3652,00145109,2025,11012,H1,ifrs-full_OtherNoncurrentFinancialLiabilities,BS,재무상태표,기타비유동금융부채,제 103 기 반기말,1.773533e+10,2025-06-30,000100
3653,00145109,2025,11012,H1,ifrs-full_OtherNoncurrentFinancialAssets,BS,재무상태표,기타비유동금융자산,제 103 기 반기말,2.816667e+07,2025-06-30,000100
3654,00145109,2025,11012,H1,ifrs-full_OtherNoncurrentLiabilities,BS,재무상태표,기타비유동부채,제 103 기 반기말,1.128510e+09,2025-06-30,000100
...,...,...,...,...,...,...,...,...,...,...,...,...
4181,00145109,2025,11014,Q3,ifrs-full_IncreaseDecreaseThroughAcquisitionOf...,SCE,자본변동표,종속기업의 취득,제 103 기 3분기,0.000000e+00,2025-09-30,000100
4182,00145109,2025,11014,Q3,ifrs-full_ShareOfOtherComprehensiveIncomeOfAss...,SCE,자본변동표,지분법투자주식의 기타포괄손익변동,제 103 기 3분기,-4.393165e+09,2025-09-30,000100
4183,00145109,2025,11014,Q3,ifrs-full_ComprehensiveIncome,SCE,자본변동표,총포괄이익,제 103 기 3분기,0.000000e+00,2025-09-30,000100
4184,00145109,2025,11014,Q3,ifrs-full_GainsLossesOnExchangeDifferencesOnTr...,SCE,자본변동표,해외사업환산차이,제 103 기 3분기,0.000000e+00,2025-09-30,000100


In [243]:
ni_parent_df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [245]:
op_df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
65,00145109,2015,11011,FY,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 93 기,8.583737e+10,2015-12-31,000100
138,00145109,2016,11011,FY,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 94 기,9.779303e+10,2016-12-31,000100
209,00145109,2016,11012,H1,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 94 기 반기,2.462414e+10,2016-06-30,000100
280,00145109,2016,11013,Q1,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 94 기 1분기,1.990684e+10,2016-03-31,000100
353,00145109,2016,11014,Q3,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 94 기 3분기,2.523724e+10,2016-09-30,000100
440,00145109,2017,11011,FY,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 95 기,8.870962e+10,2017-12-31,000100
514,00145109,2017,11012,H1,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 95 기 반기,2.074235e+10,2017-06-30,000100
585,00145109,2017,11013,Q1,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 95 기 1분기,3.551580e+10,2017-03-31,000100
668,00145109,2017,11014,Q3,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 95 기 3분기,2.197296e+10,2017-09-30,000100
757,00145109,2018,11011,FY,dart_OperatingIncomeLoss,CIS,포괄손익계산서,영업이익,제 96 기,5.012644e+10,2018-12-31,000100


In [247]:
ni_total_df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [183]:
unique_ids = df["account_id"].dropna().unique().tolist()

# 1) account_id 에서 'ContinuingOperations' 가 들어간 계정만 필터
continuing_ops_ids = [
    acc for acc in unique_ids
    if "ContinuingOperations" in acc
]

print("계속사업 관련 account_id 목록:")
print(continuing_ops_ids)

계속사업 관련 account_id 목록:
['ifrs_BasicEarningsLossPerShareFromContinuingOperations', 'ifrs_ProfitLossFromContinuingOperations', 'ifrs_IncomeTaxExpenseContinuingOperations', 'ifrs-full_IncomeTaxExpenseContinuingOperations']


In [145]:
is_table.columns.tolist()

['관계기업 기타포괄손익지분',
 '관계기업 및 공동기업투자',
 '관계기업에 대한 투자자산의 취득',
 '관계기업의 기타포괄손익에 대한 지분',
 '관계기업투자의 처분',
 '관계기업투자의 취득',
 '관계기업투자주식의 처분',
 '관계기업투자주식의 취득',
 '금융비용',
 '금융수익',
 '기말 현금및현금성자산',
 '기말현금및현금성자산',
 '기본주당반기순이익',
 '기본주당반기순이익(손실)',
 '기본주당분기순이익',
 '기본주당분기순이익(손실)',
 '기본주당순이익',
 '기본주당이익(손실)',
 '기본주당이익(원)',
 '기초 현금및현금성자산',
 '기초자본',
 '기초현금및현금성자산',
 '기타',
 '기타 비유동 부채',
 '기타 유동부채',
 '기타 투자활동으로 인한 현금유출입',
 '기타금융부채',
 '기타금융자산',
 '기타금융자산의 감소',
 '기타금융자산의 증가',
 '기타금융자산의 처분',
 '기타금융자산의 처분(취득)',
 '기타금융자산의 취득',
 '기타비유동부채',
 '기타비유동자산',
 '기타수취채권',
 '기타수취채권의 감소',
 '기타수취채권의 증가',
 '기타영업외비용',
 '기타영업외수익',
 '기타유동부채',
 '기타유동자산',
 '기타자본',
 '기타지급채무',
 '기타포괄손익',
 '기타포괄손익누계액',
 '단기금융상품',
 '단기금융상품의 감소',
 '단기금융상품의 순증감',
 '단기금융상품의 증가',
 '단기미지급금',
 '단기차입금 및 유동성 장기차입금',
 '단기투자자산',
 '단기투자자산의 순증감',
 '당기법인세부채',
 '당기법인세자산',
 '당기손익으로 재분류되는 세후기타포괄손익',
 '당기손익으로 재분류되지 않는 세후기타포괄손익',
 '당기순이익',
 '당기순이익(손실)',
 '리스부채',
 '리스부채의 상환',
 '매각예정부채',
 '매각예정자산',
 '매각예정자산의 처분',
 '매도가능금융자산',
 '매도가능금융자산의 처분',
 '매도가능금융자산의 취득',
 '매도가능금융자산평가손익',
 '매입

In [150]:
is_table.columns[is_table.columns.str.contains('기타')]

Index(['관계기업 기타포괄손익지분', '관계기업의 기타포괄손익에 대한 지분', '기타', '기타 비유동 부채', '기타 유동부채',
       '기타 투자활동으로 인한 현금유출입', '기타금융부채', '기타금융자산', '기타금융자산의 감소', '기타금융자산의 증가',
       '기타금융자산의 처분', '기타금융자산의 처분(취득)', '기타금융자산의 취득', '기타비유동부채', '기타비유동자산',
       '기타수취채권', '기타수취채권의 감소', '기타수취채권의 증가', '기타영업외비용', '기타영업외수익', '기타유동부채',
       '기타유동자산', '기타자본', '기타지급채무', '기타포괄손익', '기타포괄손익누계액',
       '당기손익으로 재분류되는 세후기타포괄손익', '당기손익으로 재분류되지 않는 세후기타포괄손익', '법인세차감후 기타포괄손익',
       '지분법 적용대상 관계기업과 공동기업의 기타포괄손익에 대한 지분(세후기타포괄손익)',
       '해외사업장환산외환차이(세후기타포괄손익)'],
      dtype='object', name='account_nm')

In [188]:
mask_name = df["account_nm"].astype(str).str.contains("계속", na=False) | \
            df["account_nm"].astype(str).str.contains("이익", na=False)

In [252]:
cont_ops_df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [195]:
df.to_excel(r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea\fs_sample.xlsx')